# Milestone 3 - Task 3: Random Forest Model

## Objective

Train and evaluate a Random Forest classifier to predict whether a Kickstarter campaign will succeed or fail.

The model will use the shared model-ready features and train/test split prepared in Task 1. Several reasonable Random Forest parameters will be tested, and the selected model will be evaluated using Accuracy, Precision, Recall, F1 Score, ROC-AUC, and a confusion matrix.

## Data Preparation

We use the same six launch-time features, preprocessing steps, and 80/20 stratified train-test split established in Milestone 3 Task 1 to ensure consistent model evaluation.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load dataset
df = pd.read_csv("../data/kickstarter_us_filtered.csv")

# Define target: successful = 1, failed = 0
df["success"] = (df["status"] == "successful").astype(int)

# Extract state from location
df["state"] = df["location"].str.rsplit(",", n=1).str[-1].str.strip()

# Use the same features as Task 1
numerical_features = ["goal", "duration", "levels"]
categorical_features = ["category", "subcategory", "state"]

X = df[numerical_features + categorical_features].copy()
y = df["success"].copy()

# Same 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Same preprocessing
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

Training shape: (30803, 119)
Testing shape: (7701, 119)


## Baseline Random Forest Model

A baseline Random Forest classifier is trained using 100 decision trees and a fixed random state for reproducibility. This provides an initial model whose performance can later be compared against tuned Random Forest configurations.

In [2]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the baseline Random Forest classifier
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train the model
rf_baseline.fit(X_train_processed, y_train)

print("Baseline Random Forest training complete!")

Baseline Random Forest training complete!


## Baseline Model Evaluation

The baseline Random Forest is evaluated on the held-out test dataset using Accuracy, Precision, Recall, F1 Score, ROC-AUC, and a confusion matrix. These metrics will help assess classification performance and provide a reference for later hyperparameter tuning.

In [3]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predict outcomes for unseen test campaigns
y_pred = rf_baseline.predict(X_test_processed)

# Predicted probability of success
y_prob = rf_baseline.predict_proba(X_test_processed)[:, 1]

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Baseline Random Forest Performance")
print("----------------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Baseline Random Forest Performance
----------------------------------
Accuracy:  0.6793
Precision: 0.6987
Recall:    0.7279
F1 Score:  0.7130
ROC-AUC:   0.7311

Confusion Matrix:
[[2163 1323]
 [1147 3068]]

Classification Report:
              precision    recall  f1-score   support

           0       0.65      0.62      0.64      3486
           1       0.70      0.73      0.71      4215

    accuracy                           0.68      7701
   macro avg       0.68      0.67      0.67      7701
weighted avg       0.68      0.68      0.68      7701



## Hyperparameter Tuning

To improve the baseline Random Forest, several configurations are evaluated using cross-validation on the training data. Parameters include the number of trees, maximum tree depth, and minimum samples required at a leaf node.

The test dataset is reserved for final evaluation to avoid selecting model parameters based on test performance.

In [4]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", 0.5]
}

rf_tuning = RandomForestClassifier(
    random_state=42,
    n_jobs=1
)

random_search = RandomizedSearchCV(
    estimator=rf_tuning,
    param_distributions=param_grid,
    n_iter=12,
    cv=3,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train_processed, y_train)

print("Best parameters:", random_search.best_params_)
print("Best CV ROC-AUC:", round(random_search.best_score_, 4))

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best parameters: {'n_estimators': 100, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}
Best CV ROC-AUC: 0.749


## Tuned Random Forest Evaluation

The best Random Forest configuration identified through cross-validation is evaluated on the held-out test dataset using the same metrics as the baseline model.

In [6]:
# Best model from hyperparameter tuning
rf_tuned = random_search.best_estimator_

# Predict on the held-out test set
y_pred_tuned = rf_tuned.predict(X_test_processed)
y_prob_tuned = rf_tuned.predict_proba(X_test_processed)[:, 1]

# Calculate metrics
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
tuned_precision = precision_score(y_test, y_pred_tuned)
tuned_recall = recall_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned)
tuned_roc_auc = roc_auc_score(y_test, y_prob_tuned)

print("Tuned Random Forest Performance")
print("--------------------------------")
print(f"Accuracy:  {tuned_accuracy:.4f}")
print(f"Precision: {tuned_precision:.4f}")
print(f"Recall:    {tuned_recall:.4f}")
print(f"F1 Score:  {tuned_f1:.4f}")
print(f"ROC-AUC:   {tuned_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

Tuned Random Forest Performance
--------------------------------
Accuracy:  0.6964
Precision: 0.7011
Recall:    0.7763
F1 Score:  0.7368
ROC-AUC:   0.7553

Confusion Matrix:
[[2091 1395]
 [ 943 3272]]


In [7]:
comparison_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Baseline Random Forest": [
        accuracy, precision, recall, f1, roc_auc
    ],
    "Tuned Random Forest": [
        tuned_accuracy, tuned_precision, tuned_recall,
        tuned_f1, tuned_roc_auc
    ]
})

comparison_df["Improvement"] = (
    comparison_df["Tuned Random Forest"]
    - comparison_df["Baseline Random Forest"]
)

comparison_df.round(4)

,Metric,Baseline Random Forest,Tuned Random Forest,Improvement
0,Accuracy,0.6793,0.6964,0.0171
1,Precision,0.6987,0.7011,0.0024
2,Recall,0.7279,0.7763,0.0484
3,F1 Score,0.7130,0.7368,0.0238
4,ROC-AUC,0.7311,0.7553,0.0243


## Extended Hyperparameter Tuning

An extended randomized search is performed to explore additional Random Forest configurations. Preprocessing is incorporated into a scikit-learn Pipeline to ensure that transformations are fitted independently within each cross-validation fold.

Model configurations are selected using cross-validated ROC-AUC on the training data, while the held-out test set is reserved for final evaluation.

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

# Combine preprocessing and Random Forest into one pipeline
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42,
        n_jobs=1
    ))
])

# Expanded hyperparameter search space
param_grid_extended = {
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__max_depth": [10, 15, 20, 30, None],
    "classifier__min_samples_split": [2, 5, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 4, 8],
    "classifier__max_features": ["sqrt", "log2", 0.5],
    "classifier__class_weight": [None, "balanced"]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

extended_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_grid_extended,
    n_iter=25,
    cv=cv,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Important: use raw X_train, not X_train_processed
extended_search.fit(X_train, y_train)

print("Best parameters:")
print(extended_search.best_params_)

print("\nBest cross-validation ROC-AUC:")
print(round(extended_search.best_score_, 4))

Fitting 3 folds for each of 25 candidates, totalling 75 fits
Best parameters:
{'classifier__n_estimators': 500, 'classifier__min_samples_split': 20, 'classifier__min_samples_leaf': 1, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 30, 'classifier__class_weight': None}

Best cross-validation ROC-AUC:
0.7538


## Extended Tuned Model Evaluation

The best model from the extended hyperparameter search is evaluated on the held-out test set using Accuracy, Precision, Recall, F1 Score, ROC-AUC, and a confusion matrix.

In [9]:
# Best model from extended tuning
rf_extended = extended_search.best_estimator_

# Predict using raw X_test because preprocessing is inside the pipeline
y_pred_extended = rf_extended.predict(X_test)
y_prob_extended = rf_extended.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
extended_accuracy = accuracy_score(y_test, y_pred_extended)
extended_precision = precision_score(y_test, y_pred_extended)
extended_recall = recall_score(y_test, y_pred_extended)
extended_f1 = f1_score(y_test, y_pred_extended)
extended_roc_auc = roc_auc_score(y_test, y_prob_extended)

print("Extended Tuned Random Forest Performance")
print("----------------------------------------")
print(f"Accuracy:  {extended_accuracy:.4f}")
print(f"Precision: {extended_precision:.4f}")
print(f"Recall:    {extended_recall:.4f}")
print(f"F1 Score:  {extended_f1:.4f}")
print(f"ROC-AUC:   {extended_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_extended))

Extended Tuned Random Forest Performance
----------------------------------------
Accuracy:  0.7028
Precision: 0.7024
Recall:    0.7929
F1 Score:  0.7449
ROC-AUC:   0.7606

Confusion Matrix:
[[2070 1416]
 [ 873 3342]]


In [10]:
comparison_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Baseline RF": [
        accuracy, precision, recall, f1, roc_auc
    ],
    "First Tuned RF": [
        tuned_accuracy, tuned_precision, tuned_recall,
        tuned_f1, tuned_roc_auc
    ],
    "Extended Tuned RF": [
        extended_accuracy, extended_precision, extended_recall,
        extended_f1, extended_roc_auc
    ]
})

comparison_df.round(4)

,Metric,Baseline RF,First Tuned RF,Extended Tuned RF
0,Accuracy,0.6793,0.6964,0.7028
1,Precision,0.6987,0.7011,0.7024
2,Recall,0.7279,0.7763,0.7929
3,F1 Score,0.7130,0.7368,0.7449
4,ROC-AUC,0.7311,0.7553,0.7606


## Key Findings and Conclusion

### Model Performance

Three Random Forest configurations were evaluated using the same six launch-time features and 80/20 train-test split established in Milestone 3 Task 1.

- **Baseline Random Forest:** Achieved 67.93% accuracy, 0.7130 F1 Score, and 0.7311 ROC-AUC.
- **First Tuned Random Forest:** Improved accuracy to 69.64%, F1 Score to 0.7368, and ROC-AUC to 0.7553.
- **Extended Tuned Random Forest:** Achieved the strongest observed performance, with 70.28% accuracy, 70.24% precision, 79.29% recall, 0.7449 F1 Score, and 0.7606 ROC-AUC.

### Hyperparameter Tuning

The extended randomized search evaluated 25 configurations using three-fold cross-validation and selected the following parameters:

- `n_estimators = 500`
- `max_depth = 30`
- `min_samples_split = 20`
- `min_samples_leaf = 1`
- `max_features = "sqrt"`
- `class_weight = None`

The best cross-validation ROC-AUC was 0.7538.

### Interpretation

Hyperparameter tuning improved the model's performance compared with the baseline. The extended model increased accuracy by approximately 2.35 percentage points and recall by 6.50 percentage points.

The model correctly identified 3,342 successful campaigns and 2,070 failed campaigns. However, it incorrectly classified 1,416 failed campaigns as successful and missed 873 successful campaigns.

The extended Random Forest demonstrated the strongest observed overall performance among the three configurations, particularly in identifying successful campaigns.

### Limitations and Next Steps

The model used only six launch-time features to reduce potential data leakage. Additional launch-time feature engineering may improve predictive performance.

Although the extended model improved over the baseline, its accuracy remains below the project's original target of approximately 80%.

The test set was examined during development, so the reported test metrics should be treated as development-stage results rather than a fully independent final evaluation.

The extended Random Forest is the leading candidate from Task 3 and will be compared with the other classification models in Milestone 3 Task 6.